# Posterior Tutorial

This tutorial demonstrates how `ls_bayesian`'s posterior subpackage wires its three independent
components together, and works through the mathematics behind the quantities a
`LogPosterior` evaluates. To keep the tutorial self-contained and fast to run, we use lightweight
dummy implementations for the forward map and the prior instead of a PDE forward solver or an
`SPDEPrior`, and the package's own concrete
`GaussianLogLikelihood`, which needs no external dependencies. Because the posterior only depends
on the `interfaces` module, any component satisfying the respective abstract base class -- dummy or
real -- wires up in exactly the same way.

## Mathematical Formulation

For a Bayesian inverse problem with parameter $m$, `ls_bayesian` models the negative log-posterior
as

$$
\begin{equation*}
    J(m) = \Phi(F(m)) + R(m),
\end{equation*}
$$

where:

- $F$ is the parameter-to-solution map, typically the solution operator of a PDE. It is usually the
  most expensive part of a posterior evaluation.
- $\Phi(u)$ is the negative log-likelihood of the solution $u = F(m)$ given observed data. For
  additive Gaussian noise $\eta \sim \mathcal{N}(0, \Gamma)$ and a linear observation operator
  $\mathcal{B}$, this is the quadratic misfit
  $\Phi(u) = \frac{1}{2}(\mathcal{B}u - d)^T \Gamma^{-1} (\mathcal{B}u - d)$.
- $R(m)$ is the negative log-density of a Gaussian prior measure $\mathcal{N}(\overline{m},
  \mathcal{C})$, $R(m) = \frac{1}{2}(m - \overline{m})^T \mathcal{C}^{-1} (m - \overline{m})$.

Differentiating $J$ with respect to $m$ gives, by the chain rule,

$$
\begin{equation*}
    \nabla_m J(m) = (\nabla_m F(m))^T \nabla_u \Phi(u) + \nabla_m R(m), \qquad u = F(m).
\end{equation*}
$$

Evaluating $u = F(m)$ is the expensive step, and it is needed for both the cost and the gradient.
`LogPosterior` therefore caches it (and the likelihood gradient $\nabla_u \Phi(u)$) for the most
recently seen parameter vector, so that a cost-then-gradient evaluation at the same $m$ -- as most
optimizers perform per iteration -- solves the forward problem only once. We verify this caching
behavior below.

## Imports and Configuration

We import NumPy and SciPy sparse for the linear algebra, SciPy's `minimize` for the closing MAP
estimation, `typing.override` for the dummy implementations, and the `interfaces`, `likelihood`,
and `posterior` modules of `ls_bayesian`'s posterior subpackage. All randomness is seeded for
reproducibility.

In [ ]:
from typing import override

import numpy as np
from scipy.optimize import minimize

from ls_bayesian.posterior import interfaces, likelihood, posterior

rng = np.random.default_rng(0)
PARAMETER_DIM = 3
NUM_VERTICES = 6
OBSERVED_VERTEX_INDICES = np.array([0, 1, 3, 5], dtype=np.int64)

## Dummy Parameter-to-Solution Map

In practice, $F$ solves a PDE for the given parameter. For this tutorial we replace it with a
linear map $F(m) = Am$, so that $\nabla_m F(m) = A$ is constant: its transpose and Jacobian-vector
products reduce to matrix-vector products, and the second-order contribution of $F$ to the
posterior Hessian vanishes. The dummy still satisfies the full `ParameterToSolutionMap` interface,
and counts its own forward evaluations so we can inspect the posterior's caching behavior later.

In [ ]:
class LinearForwardMap(interfaces.ParameterToSolutionMap):
    """Linear forward map F(m) = A m, counting its forward evaluations."""

    def __init__(self, matrix: np.ndarray) -> None:
        self.matrix = matrix
        self.num_forward_evaluations = 0

    @override
    def evaluate_forward(self, parameter_vector: np.ndarray) -> np.ndarray:
        self.num_forward_evaluations += 1
        return self.matrix @ parameter_vector

    @override
    def evaluate_gradient(
        self, solution_vector: np.ndarray, parameter_vector: np.ndarray, adjoint_vector: np.ndarray
    ) -> np.ndarray:
        return self.matrix.T @ adjoint_vector

    @override
    def evaluate_jacobian_vector_product(
        self,
        solution_vector: np.ndarray,
        parameter_vector: np.ndarray,
        direction_vector: np.ndarray,
    ) -> np.ndarray:
        return self.matrix @ direction_vector

    @override
    def evaluate_hessian_vector_product(
        self,
        solution_vector: np.ndarray,
        parameter_vector: np.ndarray,
        direction_vector: np.ndarray,
        adjoint_vector: np.ndarray,
        gradient_vector: np.ndarray,
    ) -> np.ndarray:
        return np.zeros_like(parameter_vector)


forward_map = LinearForwardMap(rng.standard_normal((NUM_VERTICES, PARAMETER_DIM)))

## Dummy Gaussian Prior

The concrete Gaussian prior shipped with `ls_bayesian` is `SPDEPrior`, which realizes $\mathcal{C}$
implicitly through finite element operators on a mesh. For this tutorial, we instead define a
small, dense Gaussian prior directly from a mean vector $\overline{m}$ and an explicit SPD
precision matrix $P = \mathcal{C}^{-1}$, obtaining the covariance $\mathcal{C} = P^{-1}$ and its
Cholesky factorization $\widehat{\mathcal{C}}\widehat{\mathcal{C}}^T = \mathcal{C}$ for sampling.

In [ ]:
class DenseGaussianPrior(interfaces.GaussianPrior):
    """Gaussian prior with an explicit, dense precision matrix P and mean m_bar."""

    def __init__(self, mean_vector: np.ndarray, precision_matrix: np.ndarray, seed: int) -> None:
        self.mean_vector = mean_vector
        self.precision_matrix = precision_matrix
        self.covariance_matrix = np.linalg.inv(precision_matrix)
        self.covariance_factor = np.linalg.cholesky(self.covariance_matrix)
        self._rng = np.random.default_rng(seed)

    @property
    @override
    def random_vector_size(self) -> int:
        return self.covariance_factor.shape[1]

    @override
    def evaluate_cost(self, parameter_vector: np.ndarray) -> float:
        difference_vector = parameter_vector - self.mean_vector
        return float(0.5 * difference_vector @ self.precision_matrix @ difference_vector)

    @override
    def evaluate_gradient(self, parameter_vector: np.ndarray) -> np.ndarray:
        return self.precision_matrix @ (parameter_vector - self.mean_vector)

    @override
    def evaluate_hessian_vector_product(self, direction_vector: np.ndarray) -> np.ndarray:
        return self.precision_matrix @ direction_vector

    @override
    def generate_sample(self) -> np.ndarray:
        random_vector = self._rng.standard_normal(self.random_vector_size)
        return self.mean_vector + self.apply_covariance_factorization(random_vector)

    @override
    def apply_covariance_operator(self, parameter_vector: np.ndarray) -> np.ndarray:
        return self.covariance_matrix @ parameter_vector

    @override
    def apply_covariance_factorization(self, random_vector: np.ndarray) -> np.ndarray:
        return self.covariance_factor @ random_vector

    @override
    def apply_precision_operator(self, parameter_vector: np.ndarray) -> np.ndarray:
        return self.precision_matrix @ parameter_vector


precision_factor = rng.standard_normal((PARAMETER_DIM, PARAMETER_DIM))
precision_matrix = precision_factor @ precision_factor.T + PARAMETER_DIM * np.eye(PARAMETER_DIM)
prior = DenseGaussianPrior(rng.standard_normal(PARAMETER_DIM), precision_matrix, seed=1)

## Concrete Gaussian Likelihood

Unlike the forward map and the prior, `ls_bayesian` already ships a concrete `Likelihood`:
`GaussianLogLikelihood`, so no dummy is needed here. We use its `from_vertex_observations` factory
for point observations at a subset of the solution's `NUM_VERTICES` degrees of freedom, with
independent Gaussian noise per observation.

In [ ]:
observation_settings = likelihood.VertexObservationSettings(
    data_vector=rng.standard_normal(OBSERVED_VERTEX_INDICES.shape[0]),
    num_vertices=NUM_VERTICES,
    observed_vertex_indices=OBSERVED_VERTEX_INDICES,
    precision_values=rng.uniform(0.5, 2.0, OBSERVED_VERTEX_INDICES.shape[0]),
)
gaussian_likelihood = likelihood.GaussianLogLikelihood.from_vertex_observations(
    observation_settings
)

## Wire the Components into a LogPosterior

`LogPosterior` only depends on the three interfaces above, so it wires up identically regardless
of whether a component is a dummy or a production implementation such as `SPDEPrior`:

In [ ]:
log_posterior = posterior.LogPosterior(gaussian_likelihood, forward_map, prior)

## Evaluate Posterior Quantities

`evaluate_cost` and `evaluate_gradient` return $J(m)$ and $\nabla_m J(m)$; `evaluate_cost_components`
and `evaluate_gradient_components` instead return the likelihood and prior contributions
separately, $(\Phi(F(m)), R(m))$ and the corresponding two gradient terms.

In [ ]:
test_parameter = rng.standard_normal(PARAMETER_DIM)

likelihood_cost, prior_cost = log_posterior.evaluate_cost_components(test_parameter)
total_cost = log_posterior.evaluate_cost(test_parameter)
likelihood_gradient, prior_gradient = log_posterior.evaluate_gradient_components(test_parameter)
total_gradient = log_posterior.evaluate_gradient(test_parameter)

print(
    f"likelihood cost: {likelihood_cost:.6f}, prior cost: {prior_cost:.6f}, total: {total_cost:.6f}"
)
print(f"gradient norm: {np.linalg.norm(total_gradient):.6f}")

## Caching Across Evaluations

Cost and gradient evaluations at the same parameter vector both need $u = F(m)$; the posterior's
internal `EvaluationCache` stores it (and the likelihood gradient) for the most recently seen
parameter vector, so `forward_map.evaluate_forward` runs only once per distinct parameter vector,
no matter how many posterior quantities are requested at it. We see this directly through the
dummy forward map's evaluation counter: the four `evaluate_cost`/`evaluate_gradient` calls above,
all at `test_parameter`, cause a single forward evaluation. A new parameter vector triggers exactly
one more.

In [ ]:
evaluations_at_test_parameter = forward_map.num_forward_evaluations
log_posterior.evaluate_gradient(test_parameter)  # cache hit, no new forward evaluation
assert forward_map.num_forward_evaluations == evaluations_at_test_parameter

other_parameter = rng.standard_normal(PARAMETER_DIM)
log_posterior.evaluate_cost(other_parameter)  # different parameter vector, cache miss
assert forward_map.num_forward_evaluations == evaluations_at_test_parameter + 1

print(f"forward evaluations so far: {forward_map.num_forward_evaluations}")

## Verifying the Gradient with Finite Differences

Since the wiring composes exact analytic derivatives from each component via the chain rule, a
central finite-difference approximation of $\nabla_m J(m)$ should agree with `evaluate_gradient` to
$O(h^2)$.

In [ ]:
def central_difference_gradient(function, point: np.ndarray, step: float) -> np.ndarray:
    gradient = np.zeros_like(point)
    for i in range(point.shape[0]):
        perturbation = np.zeros_like(point)
        perturbation[i] = step
        gradient[i] = (function(point + perturbation) - function(point - perturbation)) / (2 * step)
    return gradient


finite_difference_gradient = central_difference_gradient(
    log_posterior.evaluate_cost, test_parameter, step=1e-6
)
analytic_gradient = log_posterior.evaluate_gradient(test_parameter)
relative_error = np.linalg.norm(finite_difference_gradient - analytic_gradient) / np.linalg.norm(
    analytic_gradient
)
print(f"relative gradient error: {relative_error:.3e}")

## Finding the MAP Estimate

With cost and gradient wired up, `log_posterior` can be handed to any gradient-based optimizer to
compute the maximum a posteriori (MAP) estimate $m_{MAP} = \arg\min_m J(m)$. We use SciPy's
L-BFGS-B implementation, which calls `evaluate_cost` and `evaluate_gradient` separately at each
iterate; caching (see above) ensures each iterate's forward solve is still only performed once.

In [ ]:
forward_evaluations_before_optimization = forward_map.num_forward_evaluations

optimization_result = minimize(
    log_posterior.evaluate_cost,
    x0=np.zeros(PARAMETER_DIM),
    jac=log_posterior.evaluate_gradient,
    method="L-BFGS-B",
)

map_estimate = optimization_result.x
forward_evaluations_during_optimization = (
    forward_map.num_forward_evaluations - forward_evaluations_before_optimization
)
print(f"converged: {optimization_result.success}, iterations: {optimization_result.nit}")
print(f"forward evaluations during optimization: {forward_evaluations_during_optimization}")
print(f"MAP estimate: {map_estimate}")

## Verifying the MAP Estimate

`optimization_result.success` only reports that L-BFGS-B's own stopping criterion triggered, not
that the point found is actually the minimizer of $J$. Here we can do better: because the forward
map is linear, $F(m) = Am$, and both $\Phi$ and $R$ are quadratic, $J(m) = \Phi(F(m)) + R(m)$ is an
exact convex quadratic in $m$ -- this is a linear-Gaussian inverse problem. Its Hessian

$$
\begin{equation*}
    \mathcal{H} = A^T \mathcal{B}^T \Gamma^{-1} \mathcal{B} A + P
\end{equation*}
$$

is constant and SPD by construction (a PSD term plus the SPD prior precision $P$), so $J$ is
strictly convex and any stationary point is provably the *unique global* minimizer -- there is no
risk of L-BFGS-B having converged to a local, non-global optimum. Setting $\nabla_m J(m) = 0$ and
solving for $m$ gives the normal equations

$$
\begin{equation*}
    \mathcal{H}\, m_{MAP} = A^T \mathcal{B}^T \Gamma^{-1} d + P\overline{m},
\end{equation*}
$$

which we solve directly below, independently of `LogPosterior` and the optimizer, as a genuine
check on `optimization_result.x` (rather than re-deriving what L-BFGS-B already computed). This
closed-form solution is also exactly the posterior mean of the linear-Gaussian model.

In [ ]:
observation_matrix = likelihood.assemble_vertex_observation_matrix(
    NUM_VERTICES, OBSERVED_VERTEX_INDICES
).toarray()
noise_precision_matrix = likelihood.assemble_diagonal_precision_matrix(
    observation_settings.precision_values
).toarray()

posterior_hessian = (
    forward_map.matrix.T
    @ observation_matrix.T
    @ noise_precision_matrix
    @ observation_matrix
    @ forward_map.matrix
    + prior.precision_matrix
)
posterior_hessian_rhs = (
    forward_map.matrix.T
    @ observation_matrix.T
    @ noise_precision_matrix
    @ observation_settings.data_vector
    + prior.precision_matrix @ prior.mean_vector
)
closed_form_map_estimate = np.linalg.solve(posterior_hessian, posterior_hessian_rhs)

hessian_is_spd = bool(np.all(np.linalg.eigvalsh(posterior_hessian) > 0))
gradient_at_closed_form_estimate_norm = np.linalg.norm(
    log_posterior.evaluate_gradient(closed_form_map_estimate)
)
map_estimate_relative_error = np.linalg.norm(
    map_estimate - closed_form_map_estimate
) / np.linalg.norm(closed_form_map_estimate)

print(f"posterior Hessian is SPD: {hessian_is_spd}")
print(f"||gradient|| at closed-form estimate: {gradient_at_closed_form_estimate_norm:.3e}")
print(f"relative error, L-BFGS-B vs. closed form: {map_estimate_relative_error:.3e}")